# Inclusive-jet JES and JER

Extract and compare the jet energy scale (Gaussian mean) and jet energy resolution (Gaussian sigma) from configurable reco- and gen-level `*InclusiveJetJES*PtEta` histograms. The projections, slice fits, central-eta normalization, systematic comparisons, and JER parameterization follow `macro/plotMcClosures.C::plotJESandJER`, `extractJESandJER`, and `plotJESandJERSyst`.

Source histograms are loaded as detached copies and remain unchanged. Fits use the response distribution in each selected pT or eta bin and skip empty or invalid slices. A separate section clones and centrally normalizes the derived JER-versus-eta histograms, then saves them as both a comparison plot and a ROOT file.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from array import array
import math
import sys

PROJECT_ROOT = Path('/Users/gnigmat/work/cms/jetAnalysis')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import ROOT
except ModuleNotFoundError:
    for path in (Path('/opt/homebrew/lib/python3.14/site-packages'),
                 Path('/opt/homebrew/Cellar/root/6.40.02_1/lib/root')):
        if path.exists() and str(path) not in sys.path:
            sys.path.insert(0, str(path))
    import ROOT

ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(0)
ROOT.TH1.AddDirectory(False)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.python.histogram_io import load_histogram, resolve_combined_file, resolve_direction_file, save_histograms
from hist_analysis.python.root_style import COLORS, set_1d_style, set_2d_style, set_legend_style, set_pad_style, save_canvas

## Configuration

The nominal scan uses the configured eta intervals and pT intervals from 30 to 1000 GeV. Systematic and two-dimensional projections versus pT use `SYSTEMATIC_ETA_RANGE`, while projections versus eta use `SYSTEMATIC_PT_RANGE`. Set `RUN_SYSTEMATICS = False` to skip only the systematic-comparison section. PDFs are always written beneath `OUTPUT_DIR`; systematics-versus-eta filenames include `_pt_<low>_<high>` from the configured pT interval, and `SAVE_PNG = True` also writes PNG files.

In [ ]:
GENERATOR = 'embedding'       # embedding or pythia
DIRECTION = 'Pbgoing'          # pgoing, Pbgoing, or combined
FILE_STEM = 'jetId'
ETA_RANGES = (-3.0, -2.4, -1.9, -1.6, -1.3, -0.8, 0.0, 0.8, 1.3, 1.6, 1.9, 2.4, 3.0)
PT_RANGES = (30., 50., 80., 100., 120., 150., 180., 220., 280., 350., 450., 540., 1000.)
SYSTEMATIC_ETA_RANGE = (-0.8, 0.8)
SYSTEMATIC_PT_RANGE = (350., 500.)
FIT_PT_RANGE = (30., 800.)
RUN_SYSTEMATICS = True
SAVE_PNG = False
OUTPUT_DIR = PROJECT_ROOT / 'hist_analysis' / 'output' / 'jes_jer'

def mc_file(generator, direction):
    if direction == 'combined':
        return resolve_combined_file(BASE_DIR, generator, FILE_STEM)
    return resolve_direction_file(BASE_DIR, generator, direction, FILE_STEM)

INPUT_FILE = mc_file(GENERATOR, DIRECTION)
if not INPUT_FILE.exists():
    raise FileNotFoundError(INPUT_FILE)
print(INPUT_FILE)

## Projection and extraction helpers

The TH3 axes are `(response, pT, eta)`. ROOT's `Project3D('xy')` produces `(pT, response)`, while `Project3D('xz')` produces `(eta, response)`. A small inward offset prevents a boundary bin from entering both adjacent selections, matching the macro. The output histograms preserve the selected-axis bin edges.

In [ ]:
_objects = []  # retain PyROOT-owned objects for interactive redraws

def _unique(prefix):
    return f'{prefix}_{len(_objects)}'

def project_response(hist3, selection, *, versus):
    if versus not in ('pt', 'eta'):
        raise ValueError("versus must be 'pt' or 'eta'")
    axis = hist3.GetZaxis() if versus == 'pt' else hist3.GetYaxis()
    low, high = selection
    axis.SetRangeUser(low + 0.001, high - 0.001)
    try:
        result = hist3.Project3D('xy' if versus == 'pt' else 'xz')
        result.SetName(_unique(f'{hist3.GetName()}_{versus}'))
        result.SetDirectory(0)
    finally:
        axis.SetRange(0, 0)
    if result.GetEntries() <= 0 or result.Integral() == 0:
        raise ValueError(f'Empty projection of {hist3.GetName()} for {selection}')
    _objects.append(result)
    return result

def extract_jes_jer(response2d, *, versus, style=0, fit_jer=False):
    if versus not in ('pt', 'eta'):
        raise ValueError("versus must be 'pt' or 'eta'")
    xaxis = response2d.GetXaxis()
    edges = [xaxis.GetBinLowEdge(1)] + [xaxis.GetBinUpEdge(i) for i in range(1, xaxis.GetNbins() + 1)]
    x_title = 'p_{T}^{ref} (GeV)' if versus == 'pt' else '#eta'
    jes = ROOT.TH1D(_unique('jes'), f'JES vs. {x_title};{x_title};JES', len(edges) - 1, array('d', edges))
    jer = ROOT.TH1D(_unique('jer'), f'JER vs. {x_title};{x_title};JER', len(edges) - 1, array('d', edges))
    for hist in (jes, jer):
        hist.Sumw2()
        hist.SetDirectory(0)
        set_1d_style(hist, style)
    slice_fits = []
    for bin_x in range(1, response2d.GetNbinsX() + 1):
        projection = response2d.ProjectionY(_unique('response_slice'), bin_x, bin_x)
        projection.SetDirectory(0)
        if projection.GetEntries() <= 0:
            continue
        mean, sigma = projection.GetMean(), projection.GetRMS()
        if not math.isfinite(mean) or not math.isfinite(sigma) or sigma <= 0:
            continue
        fit = ROOT.TF1(_unique('gaus'), 'gaus', mean - 2.0 * sigma, mean + 2.0 * sigma)
        fit.SetParameters(projection.GetMaximum(), mean, sigma)
        projection.Fit(fit, 'MRQEN0')
        if all(math.isfinite(fit.GetParameter(i)) for i in (1, 2)) and fit.GetParameter(2) > 0:
            jes.SetBinContent(bin_x, fit.GetParameter(1))
            jes.SetBinError(bin_x, fit.GetParError(1))
            jer.SetBinContent(bin_x, fit.GetParameter(2))
            jer.SetBinError(bin_x, fit.GetParError(2))
        slice_fits.extend((projection, fit))
    jer_fit = None
    if fit_jer:
        jer_fit = ROOT.TF1(_unique('jer_fit'), 'sqrt([0]*[0] + [1]*[1]/x)', *FIT_PT_RANGE)
        jer_fit.SetParameters(0.002, 1.0)
        jer_fit.SetLineColor(COLORS[style])
        jer_fit.SetLineWidth(2)
        jer.Fit(jer_fit, 'MRSN0')
    _objects.extend((jes, jer, *slice_fits))
    if jer_fit:
        _objects.append(jer_fit)
    return jes, jer, jer_fit

def normalize_jer_in_eta_range(histogram, eta_range=(-0.8, 0.8)):
    normalized = histogram.Clone(_unique('jer_eta_normalized'))
    normalized.SetDirectory(0)
    axis = normalized.GetXaxis()
    selected_bins = [bin_index for bin_index in range(1, normalized.GetNbinsX() + 1)
                     if axis.GetBinLowEdge(bin_index) >= eta_range[0]
                     and axis.GetBinUpEdge(bin_index) <= eta_range[1]]
    if not selected_bins:
        raise ValueError(f'No bins are fully contained in eta range {eta_range}')
    central_sum = sum(normalized.GetBinContent(bin_index) for bin_index in selected_bins)
    if central_sum <= 0:
        raise ValueError(f'Non-positive JER sum in eta range {eta_range}')
    normalized.Scale(len(selected_bins) / central_sum)
    _objects.append(normalized)
    return normalized

def draw_overlay(curves, *, y_title, y_range, x_range, output, fits=None, annotations=()):
    external_legend = len(curves) > 7
    canvas = ROOT.TCanvas(_unique('canvas'), '', 1050 if external_legend else 900, 800)
    set_pad_style(canvas)
    if external_legend:
        canvas.SetRightMargin(0.32)
        legend = ROOT.TLegend(0.68, max(0.18, 0.87 - 0.055 * len(curves)), 0.97, 0.87)
        
    else:
        legend = ROOT.TLegend(0.55, max(0.52, 0.87 - 0.055 * len(curves)), 0.85, 0.87)
    set_legend_style(legend)
    if external_legend:
        legend.SetTextSize(0.025)
    else:
        legend.SetTextSize(0.034)
    for index, (label, hist) in enumerate(curves.items()):
        hist.GetYaxis().SetTitle(y_title)
        hist.GetYaxis().SetRangeUser(*y_range)
        hist.GetXaxis().SetRangeUser(*x_range)
        hist.Draw('E1' if index == 0 else 'E1 SAME')
        legend.AddEntry(hist, label, 'p')
        if fits and fits.get(label):
            fits[label].Draw('SAME')
    legend.Draw()
    latex = ROOT.TLatex()
    latex.SetNDC(True)
    latex.SetTextFont(42)
    latex.SetTextSize(0.034)
    direction_label = {'pgoing': 'p-going', 'Pbgoing': 'Pb-going', 'combined': 'combined'}[DIRECTION]
    for line_index, line in enumerate((GENERATOR.capitalize(), direction_label, *annotations)):
        latex.DrawLatex(0.17, 0.86 - 0.045 * line_index, line)
    canvas.Update()
    save_canvas(canvas, output, save_png=SAVE_PNG)
    _objects.extend((canvas, legend, latex))
    return canvas

def draw_response_2d(response2d, *, versus, selection_text, output):
    canvas = ROOT.TCanvas(_unique('response_canvas'), '', 900, 800)
    set_pad_style(canvas)
    canvas.SetRightMargin(0.14)
    set_2d_style(response2d)
    response2d.GetXaxis().SetTitle('p_{T}^{ref} (GeV)' if versus == 'pt' else '#eta')
    response2d.GetYaxis().SetTitle('p_{T}^{reco}/p_{T}^{ref}')
    response2d.GetYaxis().SetRangeUser(0., 2.)
    if versus == 'pt':
        response2d.GetXaxis().SetRangeUser(30., 500.)
    else:
        response2d.GetXaxis().SetRangeUser(-3., 3.)
    response2d.Draw('COLZ')
    latex = ROOT.TLatex()
    latex.SetNDC(True)
    latex.SetTextFont(42)
    latex.SetTextSize(0.034)
    direction_label = {'pgoing': 'p-going', 'Pbgoing': 'Pb-going', 'combined': 'combined'}[DIRECTION]
    for line_index, line in enumerate((GENERATOR.capitalize(), direction_label, selection_text)):
        latex.DrawLatex(0.17, 0.86 - 0.045 * line_index, line)
    canvas.Update()
    save_canvas(canvas, output, save_png=SAVE_PNG)
    _objects.extend((canvas, latex))
    return canvas

## Nominal dependence on eta and pT

This reproduces the scan in `plotJESandJER`: JES/JER versus reference pT in the configured eta intervals and versus eta in the configured pT intervals. The current pT intervals cover 30–1000 GeV. Plot legends use marker-only entries; comparisons with more than seven curves reserve a separate right margin for the legend.

In [ ]:
nominal3d = load_histogram(INPUT_FILE, 'hRecoInclusiveJetJESPtEta')
eta_scan = {}
for index, eta_range in enumerate(zip(ETA_RANGES[:-1], ETA_RANGES[1:])):
    response = project_response(nominal3d, eta_range, versus='pt')
    eta_scan[eta_range] = extract_jes_jer(response, versus='pt', style=index, fit_jer=True)

pt_scan = {}
for index, pt_range in enumerate(zip(PT_RANGES[:-1], PT_RANGES[1:])):
    response = project_response(nominal3d, pt_range, versus='eta')
    pt_scan[pt_range] = extract_jes_jer(response, versus='eta', style=index, fit_jer=False)

eta_labels = {f'{low:g} < #eta < {high:g}': values for (low, high), values in eta_scan.items()}
draw_overlay({label: values[0] for label, values in eta_labels.items()}, y_title='JES',
             y_range=(0.9, 1.1), x_range=(30., 500.),
             output=OUTPUT_DIR / f'{GENERATOR}_{DIRECTION}_JES_vs_pt_etaComparison.pdf')
draw_overlay({label: values[1] for label, values in eta_labels.items()}, y_title='JER',
             y_range=(0., 0.3), x_range=(30., 500.),
             fits={label: values[2] for label, values in eta_labels.items()},
             output=OUTPUT_DIR / f'{GENERATOR}_{DIRECTION}_JER_vs_pt_etaComparison.pdf')

pt_labels = {f'{low:g} < p_{{T}} < {high:g} GeV': values for (low, high), values in pt_scan.items()}
draw_overlay({label: values[0] for label, values in pt_labels.items()}, y_title='JES',
             y_range=(0.9, 1.1), x_range=(-3.0, 3.0),
             output=OUTPUT_DIR / f'{GENERATOR}_{DIRECTION}_JES_vs_eta_ptComparison.pdf')
draw_overlay({label: values[1] for label, values in pt_labels.items()}, y_title='JER',
             y_range=(0., 0.3), x_range=(-3.0, 3.0),
             output=OUTPUT_DIR / f'{GENERATOR}_{DIRECTION}_JER_vs_eta_ptComparison.pdf')

## Centrally normalized JER versus eta

Each pT projection is cloned and scaled so that the arithmetic mean of the JER bins fully contained in `-0.8 < eta < 0.8` is one, following `plotMcClosures.C::plotJESandJER`. The unnormalized JER histograms above remain unchanged. The normalized comparison is saved as a PDF, and every normalized histogram is written with a pT-range-specific key to `<generator>_<direction>_JER_vs_eta_ptComparison_normalized.root` in `OUTPUT_DIR`.

In [ ]:
normalized_jer_vs_eta = {}
for (pt_low, pt_high), values in pt_scan.items():
    label = f'{pt_low:g} < p_{{T}} < {pt_high:g} GeV'
    histogram = normalize_jer_in_eta_range(values[1])
    low_tag = f'{pt_low:g}'.replace('-', 'm').replace('.', 'p')
    high_tag = f'{pt_high:g}'.replace('-', 'm').replace('.', 'p')
    histogram.SetName(f'hRecoInclusiveJetJERVsEtaNormalized_pt_{low_tag}_{high_tag}')
    normalized_jer_vs_eta[label] = histogram

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NORMALIZED_JER_ROOT_FILE = OUTPUT_DIR / f'{GENERATOR}_{DIRECTION}_JER_vs_eta_ptComparison_normalized.root'
save_histograms(NORMALIZED_JER_ROOT_FILE, normalized_jer_vs_eta.values())
print(f'Saved {len(normalized_jer_vs_eta)} normalized JER histograms to {NORMALIZED_JER_ROOT_FILE}')

draw_overlay(
    normalized_jer_vs_eta,
    y_title='Normalized JER', y_range=(0.7, 1.4), x_range=(-3.0, 3.0),
    annotations=('-0.8 < #eta < 0.8 normalization',),
    output=OUTPUT_DIR / f'{GENERATOR}_{DIRECTION}_JER_vs_eta_ptComparison_normalized.pdf',
)

## JER systematic comparisons

The active `SYSTEMATICS` mapping currently compares reco jets without smearing, reco jets with the nominal x1 smearing, gen jets with eta-dependent nominal smearing, and gen jets with nominal x1 smearing. The reference nominal/up/down/double-SF alternatives remain available as commented entries and can be enabled explicitly. For every active entry, the notebook plots both JES and JER versus pT and eta; this keeps possible scale shifts visible alongside resolution changes.

In [ ]:
SYSTEMATICS = {
    'reco (no smearing)': ('hRecoInclusiveJetJESPtEta', 2),
    # 'JER nominal': ('hRecoInclusiveJetJESDefPtEta', 0),
    # 'JER up': ('hRecoInclusiveJetJESUpPtEta', 1),
    # 'JER down': ('hRecoInclusiveJetJESDownPtEta', 3),
    'reco (nominal, x1)': ('hRecoInclusiveJetJESDefNoSFPtEta', 6),
    'gen (nominal, #eta-dep)': ('hGenInclusiveJetJESDefExtraPtEta', 5),
    'gen (nominal, x1)': ('hGenInclusiveJetJESDefPtEta', 4),
    # 'nominal, double SF': ('hRecoInclusiveJetJESDefDoublePtEta', 7),
}

systematic_pt = {}
systematic_eta = {}
if RUN_SYSTEMATICS:
    for label, (key, style) in SYSTEMATICS.items():
        hist3 = load_histogram(INPUT_FILE, key)
        response_pt = project_response(hist3, SYSTEMATIC_ETA_RANGE, versus='pt')
        systematic_pt[label] = extract_jes_jer(response_pt, versus='pt', style=style, fit_jer=True)
        response_eta = project_response(hist3, SYSTEMATIC_PT_RANGE, versus='eta')
        systematic_eta[label] = extract_jes_jer(response_eta, versus='eta', style=style, fit_jer=False)

    tag = f'{GENERATOR}_{DIRECTION}'
    draw_overlay({label: values[0] for label, values in systematic_pt.items()}, y_title='JES',
                 y_range=(0.9, 1.1), x_range=(30., 500.),
                 annotations=(f'{SYSTEMATIC_ETA_RANGE[0]:g} < #eta < {SYSTEMATIC_ETA_RANGE[1]:g}',),
                 output=OUTPUT_DIR / f'{tag}_JES_systematics_vs_pt.pdf')
    draw_overlay({label: values[1] for label, values in systematic_pt.items()}, y_title='JER',
                 y_range=(0., 0.3), x_range=(30., 500.),
                 fits={label: values[2] for label, values in systematic_pt.items()},
                 annotations=(f'{SYSTEMATIC_ETA_RANGE[0]:g} < #eta < {SYSTEMATIC_ETA_RANGE[1]:g}',),
                 output=OUTPUT_DIR / f'{tag}_JER_systematics_vs_pt.pdf')
    draw_overlay({label: values[0] for label, values in systematic_eta.items()}, y_title='JES',
                 y_range=(0.9, 1.1), x_range=(-3.0, 3.0),
                 annotations=(f'{SYSTEMATIC_PT_RANGE[0]:g} < p_{{T}} < {SYSTEMATIC_PT_RANGE[1]:g} GeV',),
                 output=OUTPUT_DIR / f'{tag}_JES_systematics_vs_eta_pt_{SYSTEMATIC_PT_RANGE[0]:g}_{SYSTEMATIC_PT_RANGE[1]:g}.pdf')
    draw_overlay({label: values[1] for label, values in systematic_eta.items()}, y_title='JER',
                 y_range=(0., 0.3), x_range=(-3.0, 3.0),
                 annotations=(f'{SYSTEMATIC_PT_RANGE[0]:g} < p_{{T}} < {SYSTEMATIC_PT_RANGE[1]:g} GeV',),
                 output=OUTPUT_DIR / f'{tag}_JER_systematics_vs_eta_pt_{SYSTEMATIC_PT_RANGE[0]:g}_{SYSTEMATIC_PT_RANGE[1]:g}.pdf')

## Fit summary

For each active systematic-comparison entry, the table reports the constant and stochastic parameters from the pT-dependent JER model `sqrt([0]^2 + [1]^2/pT)`, together with chi2/ndf. Inspect the response slices and fit quality before using these values quantitatively; as in the reference macro, finite Gaussian slice parameters are retained even when ROOT's minimizer reports a warning.

In [ ]:
fit_summary = []
for label, (_, jer, fit) in systematic_pt.items():
    fit_summary.append({
        'variation': label,
        'constant term': fit.GetParameter(0),
        'stochastic term': fit.GetParameter(1),
        'chi2/ndf': fit.GetChisquare() / fit.GetNDF() if fit.GetNDF() else float('nan'),
    })
fit_summary

## Two-dimensional response distributions

The final section draws the nominal `pT(reco)/pT(ref)` response versus reference pT for the configured `SYSTEMATIC_ETA_RANGE`, and versus eta for `SYSTEMATIC_PT_RANGE`. These COLZ maps are the two-dimensional inputs corresponding to the slice-fit selections and are saved as PDFs in `OUTPUT_DIR`.

In [ ]:
response_vs_pt = project_response(nominal3d, SYSTEMATIC_ETA_RANGE, versus='pt')
response_vs_eta = project_response(nominal3d, SYSTEMATIC_PT_RANGE, versus='eta')
tag = f'{GENERATOR}_{DIRECTION}'

draw_response_2d(
    response_vs_pt, versus='pt',
    selection_text=f'{SYSTEMATIC_ETA_RANGE[0]:g} < #eta < {SYSTEMATIC_ETA_RANGE[1]:g}',
    output=OUTPUT_DIR / f'{tag}_response_vs_pt.pdf',
)
draw_response_2d(
    response_vs_eta, versus='eta',
    selection_text=f'{SYSTEMATIC_PT_RANGE[0]:g} < p_{{T}} < {SYSTEMATIC_PT_RANGE[1]:g} GeV',
    output=OUTPUT_DIR / f'{tag}_response_vs_eta.pdf',
)